# Vues mensuelles par contribution — générique vs. personnalisées

Pour **chaque contribution** de code.travail.gouv.fr, le nombre de **vues par mois**
de la page **générique** (`/contribution/{slug}`) comparé à la **somme** des pages
**personnalisées** par convention collective (`/contribution/{idcc}-{slug}` et
`/contribution/{slug}/{idcc}`), sur **janvier → juin 2026**, avec le **cumul** sur
la période.

La liste des contributions est récupérée depuis le
[sitemap](https://code.travail.gouv.fr/sitemap.xml) ; les vues viennent de l'**API
Reporting Matomo** (`MATOMO_*` dans `.env`).

> Astuce : mettre `DEMO = True` pour un aperçu instantané sur données synthétiques,
> sans credentials ni réseau.

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import pandas as pd

from analysis.reports.contrib_monthly_views import (
    METRICS,
    aggregate_months,
    build_bar_chart,
    build_demo_data,
    build_evolution_chart,
    fetch_generic_slugs,
    fetch_live_months,
    write_csv,
)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

In [ ]:
# ── Paramètres ────────────────────────────────────────────────────────────
START = "2026-01-01"  # début de la plage (YYYY-MM-DD)
END = "2026-06-30"    # fin de la plage
METRIC = "views"      # "views" = pages vues (nb_hits) ; "visits" = visites
TOP = None            # None = toutes les contributions ; un entier = top N (ex. 15)
DEMO = False          # True = données synthétiques, sans Matomo ni sitemap

metric_field, metric_label = METRICS[METRIC]

## 1. Récupération (sitemap + Matomo)

On récupère la liste des contributions génériques depuis le sitemap, puis un bucket
de vues par mois via Matomo (`period="month"`). Le résultat brut est mis en cache
sous `../output/.cache/`. En `DEMO`, tout est synthétique et instantané.

In [ ]:
if DEMO:
    slugs, months = build_demo_data()
else:
    slugs = fetch_generic_slugs()
    months = fetch_live_months(START, END, use_cache=True)

print(f"{len(slugs)} contributions · {len(months)} mois")

## 2. Table complète — générique vs perso, par mois + cumul

Une ligne par **contribution** (toutes affichées) ; colonnes `{YYYY-MM}_generic` /
`{YYYY-MM}_perso` pour chaque mois, puis `cumul_generic` / `cumul_perso` /
`cumul_total`. Triée par vues cumulées décroissantes.

> Certaines contributions sont à **0** : leur slug *actuel* (sitemap) n'a aucune vue
> dans Matomo sur la période — page récente ou **renommée** (le trafic est alors
> comptabilisé sous l'ancien slug). Ces lignes restent dans le tableau/CSV mais sont
> **exclues du graphe en barres**.

In [ ]:
df = aggregate_months(months, metric_field, slugs)
df

## 3. Graphe — barres, toutes les contributions

Barres empilées **générique** (bleu) vs **perso Σ CC** (rouge), une par
contribution, classées par vues cumulées. Les contributions à 0 vue sont exclues
(pas de barre vide). `TOP = 15` (par ex.) limiterait aux 15 premières ;
`TOP = None` affiche toutes celles qui ont des vues.

In [ ]:
from pathlib import Path

from IPython.display import Image

out = Path("../output")
out.mkdir(parents=True, exist_ok=True)
bar = out / "contrib_monthly_bar.png"
build_bar_chart(df, metric_label, bar, top_n=TOP)
Image(str(bar))

## 4. Graphe — évolution mensuelle

Évolution mois par mois, **sommée sur toutes les contributions** : générique,
perso Σ CC, et le total — pour lire la tendance générique vs personnalisées sur la
période.

In [ ]:
evo = out / "contrib_monthly_evolution.png"
build_evolution_chart(df, metric_label, evo)
Image(str(evo))

## 5. Export CSV

In [ ]:
csv_path = out / "contrib_monthly.csv"
write_csv(df, csv_path)
print(f"✓ CSV écrit : {csv_path.resolve()}")

## Aller plus loin

- `METRIC = "visits"` pour raisonner en visites (`nb_visits`) plutôt qu'en vues.
- `TOP = 15` pour restreindre le graphe en barres aux 15 premières contributions.
- Le CSV complet contient le détail mois par mois **et** le cumul par contribution.
- Inspecter un bucket Matomo brut : `months[list(months)[0]][:5]`.